# Token Efficiency: Materi Praktik

**Road to DevFest Jakarta 2026** · Eric Julianto · Orvix.id

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/algonacci/road-to-devfest-2026-jkt/blob/main/token_efficiency.ipynb)

Notebook ini pendamping slide. Urutannya sama dengan deck, jadi bisa dibuka bergantian:

| Bagian | Slide | Butuh API key? |
|:---|:---|:---:|
| 0. Setup | Pakai di Aplikasi | ya |
| 1. Dasar: token | Teks → Token → Angka, Token Bukan Kata, Format Data | tidak |
| 2. Habisnya di input | Habisnya di Input, Contoh Agent | ya |
| 3. Prompt caching | Cara Kerjanya, Umur Cache, Atur TTL, Aturan #1 | ya |
| 4. Biaya ganti model | Ganti ke Model Frontier | tidak |
| 5. Level aplikasi | Anatomi Request, Pembunuh Cache, Cheat Code 1 | sebagian |
| 6. Agent loop | Agent Hidup dari Cache | ya |
| 7. Kuota sadar cache | Tagihan & Kuota, Kuota Tinggal 8% | tidak |

Bagian yang tidak butuh API key bisa langsung dijalankan.

## 0. Setup

API key diambil dari **platform.orvix.id/e/ROAD-TO-DEVFEST-JKT-2026**, scope `ai:invoke`. Key diketik lewat `getpass`, tidak tersimpan di notebook.

In [ ]:
!pip -q install openai tiktoken pillow matplotlib

In [ ]:
import getpass
import os

from openai import OpenAI

BASE_URL = "https://api.orvix.id/v1"
MODEL = "orvix/deepseek-v4.1-flash"

if not os.environ.get("ORVIX_API_KEY"):
    os.environ["ORVIX_API_KEY"] = getpass.getpass("API key Orvix: ")

client = OpenAI(base_url=BASE_URL, api_key=os.environ["ORVIX_API_KEY"], timeout=120)

Helper untuk membaca `usage`. Gateway OpenAI-compatible biasanya mengisi `prompt_tokens_details.cached_tokens`. DeepSeek native memakai `prompt_cache_hit_tokens`, jadi keduanya dicek.

In [ ]:
def cached_tokens(usage) -> int:
    details = getattr(usage, "prompt_tokens_details", None)
    cached = getattr(details, "cached_tokens", None) if details else None
    if cached is None:
        extra = getattr(usage, "model_extra", None) or {}
        cached = extra.get("prompt_cache_hit_tokens")
    return cached or 0


def show_usage(resp, label=""):
    u = resp.usage
    cached = cached_tokens(u)
    hit = cached / u.prompt_tokens if u.prompt_tokens else 0
    print(
        f"{label:<14} input={u.prompt_tokens:>7,}  cache={cached:>7,} ({hit:4.0%})  "
        f"output={u.completion_tokens:>5,}"
    )
    return u.prompt_tokens, cached, u.completion_tokens

In [ ]:
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Halo dari DevFest! Jawab satu kalimat."}],
)
print(reply.choices[0].message.content)
show_usage(reply, "tes pertama");

## 1. Dasar: token

Tokenizer `o200k_base` (dipakai model OpenAI terbaru). Model lain punya tokenizer sendiri, jadi angkanya bisa sedikit beda, tapi polanya sama.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")

teks = "Halo DevFest Jakarta!"
ids = enc.encode(teks)
print("teks   :", teks)
print("token  :", [enc.decode([i]) for i in ids])
print("id     :", ids)
print("jumlah :", len(ids), "token")

### Token bukan kata

In [ ]:
kata = [
    "development",
    "pengembangan",
    "accountability",
    "mempertanggungjawabkan",
    "Rp100.000",
    "2026-10-17",
]
for k in kata:
    ids = enc.encode(k)
    potongan = " · ".join(enc.decode([i]) for i in ids)
    print(f"{k:<24} {len(ids):>2} token   {potongan}")

### English vs Indonesia

Kalimat yang sama, permintaan bikin fungsi + unit test. Ganti kalimatnya sendiri dan bandingkan.

In [ ]:
en = "Write a Python function that validates an Indonesian phone number, then add unit tests for valid and invalid inputs."
id_ = "Tulis fungsi Python yang memvalidasi nomor telepon Indonesia, lalu tambahkan unit test untuk input valid dan tidak valid."

for label, s in [("English", en), ("Indonesia", id_)]:
    print(f"{label:<10} {len(s):>4} karakter  {len(enc.encode(s)):>3} token")

### Format data juga token

Data yang sama persis, tiga format. Ini juga yang ada di slide **Coba Sendiri: Format Data**.

In [ ]:
import csv
import io
import json

produk = [
    {"id": 1, "nama": "Kopi Susu Gula Aren", "kategori": "Minuman", "harga": 18000, "stok": 42},
    {"id": 2, "nama": "Nasi Goreng Kampung", "kategori": "Makanan", "harga": 25000, "stok": 15},
    {"id": 3, "nama": "Teh Tarik", "kategori": "Minuman", "harga": 12000, "stok": 30},
    {"id": 4, "nama": "Mie Ayam Bakso", "kategori": "Makanan", "harga": 22000, "stok": 8},
    {"id": 5, "nama": "Pisang Goreng", "kategori": "Camilan", "harga": 10000, "stok": 25},
]


def as_csv(rows):
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=rows[0].keys(), lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return buf.getvalue()


formats = {
    "JSON rapi": json.dumps(produk, indent=2, ensure_ascii=False),
    "JSON minified": json.dumps(produk, separators=(",", ":"), ensure_ascii=False),
    "CSV": as_csv(produk),
}

base = len(enc.encode(formats["JSON rapi"]))
for name, text in formats.items():
    n = len(enc.encode(text))
    print(f"{name:<14} {n:>4} token   {base / n:.1f}× lebih hemat dari JSON rapi")

In [ ]:
print(formats["CSV"])

## 2. Habisnya di input, bukan output

Satu request dengan system prompt panjang. Lihat perbandingan token input dan output.

In [ ]:
ATURAN = "\n".join(
    f"{i}. Aturan produk nomor {i}: jawab sopan, ringkas, pakai Bahasa Indonesia, "
    f"jangan mengarang data, sebutkan sumber kalau ada, dan tolak permintaan di luar topik toko."
    for i in range(1, 121)
)
SYSTEM = "Kamu asisten toko kopi DevFest.\n\n" + ATURAN

print("system prompt ±", len(enc.encode(SYSTEM)), "token (perkiraan tiktoken)")

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Ada promo apa hari ini?"},
    ],
)
p, c, o = show_usage(resp, "1 request")
print(f"output cuma {o / (p + o):.1%} dari total token")

## 3. Prompt caching

Kirim prefix yang **identik** beberapa kali. Request pertama = cache miss (warm-up), berikutnya = cache hit.

> Cache bersifat best-effort. Kalau `cache` masih 0, tunggu beberapa detik lalu jalankan lagi.

In [ ]:
import time

for i in range(1, 4):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Pertanyaan ke-{i}: jam buka toko?"},
        ],
    )
    show_usage(resp, f"request {i}")
    time.sleep(2)

### Aturan #1: prefix harus identik

Satu karakter berubah di **depan** → semua di belakangnya dihitung ulang. Bandingkan jam di awal system prompt vs di pesan terakhir.

In [ ]:
from datetime import datetime


def kirim(system, user):
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
    )


print("❌ jam di awal system prompt")
for i in range(3):
    now = datetime.now().strftime("%H:%M:%S.%f")
    show_usage(kirim(f"Waktu sekarang {now}.\n" + SYSTEM, "Jam buka toko?"), f"request {i + 1}")
    time.sleep(2)

print("\n✅ jam di pesan terakhir")
for i in range(3):
    now = datetime.now().strftime("%H:%M:%S.%f")
    show_usage(kirim(SYSTEM, f"Jam buka toko? (waktu: {now})"), f"request {i + 1}")
    time.sleep(2)

### Jadikan metrik

Simpan `cache_hit` per request di log aplikasi. Kalau tidak pernah dilihat, kita tidak tahu sedang bayar 1× atau 10×.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
log = logging.getLogger("llm")

u = resp.usage
cached = cached_tokens(u)
log.info("cache_hit=%.0f%% prompt=%d", cached / u.prompt_tokens * 100, u.prompt_tokens)

### Atur TTL di kode (referensi)

Snippet dari slide **Atur TTL di Kode**. Tidak dijalankan di sini karena butuh API key masing-masing provider.

**Claude**: default 5 menit, timer di-reset tiap hit; opsi 1 jam dengan biaya tulis 2×.

```python
response = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1024,
    system=[{
        "type": "text",
        "text": SYSTEM_PROMPT_PANJANG,
        "cache_control": {"type": "ephemeral"},                # TTL 5 menit (default)
        # "cache_control": {"type": "ephemeral", "ttl": "1h"}  # TTL 1 jam
    }],
    messages=[{"role": "user", "content": "Halo!"}],
)
print(response.usage.cache_creation_input_tokens)  # warm-up
print(response.usage.cache_read_input_tokens)      # cache hit
```

**Gemini** (explicit cache): TTL diatur sendiri, default 1 jam.

```python
cache = client.caches.create(
    model="gemini-3.8-flash",
    config=types.CreateCachedContentConfig(
        system_instruction=SYSTEM_PROMPT_PANJANG,
        ttl="3600s",
    ),
)
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Halo!",
    config=types.GenerateContentConfig(cached_content=cache.name),
)
client.caches.update(name=cache.name, config=types.UpdateCachedContentConfig(ttl="7200s"))
```

**DeepSeek**: tidak ada parameter TTL. Cache otomatis, dibuang kalau menganggur (biasanya beberapa jam sampai hari).

```python
print(response.usage.prompt_cache_hit_tokens)   # cache hit
print(response.usage.prompt_cache_miss_tokens)  # dihitung baru
```

## 4. Biaya ganti model di tengah sesi

Hitungan dari slide **Ganti ke Model Frontier**. Harga per 1 juta token (MTok), input saja.

In [ ]:
HARGA = {  # $/MTok: (input baru, input dari cache)
    "DeepSeek V4.1 Flash (jam sibuk)": (0.30, 0.006),
    "Claude Fable 5.1": (10.00, 0.25),
}


def biaya_input(model, prompt_tokens, cache_ratio):
    baru, cache = HARGA[model]
    return (prompt_tokens * (1 - cache_ratio) * baru + prompt_tokens * cache_ratio * cache) / 1e6


konteks = 142_750
sebelum = biaya_input("DeepSeek V4.1 Flash (jam sibuk)", konteks, 0.92)
sesudah = biaya_input("Claude Fable 5.1", konteks, 0.0)
lanjut = biaya_input("Claude Fable 5.1", konteks, 0.92)

print(f"giliran 30, DeepSeek, cache warm : ${sebelum:.4f}")
print(f"giliran 31, Fable, cache kosong  : ${sesudah:.4f}  ({sesudah / sebelum:.0f}×)")
print(f"giliran 32+, Fable, cache warm   : ${lanjut:.4f}  ({lanjut / sebelum:.0f}×)")

## 5. Level aplikasi

### Anatomi request: stabil di depan, berubah di belakang

In [ ]:
def build_messages(profil_user, riwayat, dokumen_rag, pertanyaan):
    system = SYSTEM  # instruksi produk: hampir tidak pernah berubah
    return [
        {"role": "system", "content": system},
        {"role": "system", "content": f"Profil user: {json.dumps(profil_user, sort_keys=True)}"},
        *riwayat,  # bertambah tiap giliran, cukup append
        {
            "role": "user",
            "content": (
                f"Dokumen:\n{dokumen_rag}\n\n"
                f"Waktu: {datetime.now():%H:%M}\n"
                f"Pertanyaan: {pertanyaan}"
            ),
        },
    ]


msgs = build_messages({"nama": "Budi", "kota": "Jakarta"}, [], "Promo: kopi susu diskon 20%.", "Ada promo?")
for m in msgs:
    print(m["role"], "→", m["content"][:60].replace("\n", " "), "...")

### Pembunuh cache diam-diam: urutan key JSON

`dict` yang urutannya berubah menghasilkan string berbeda → prefix berbeda.

In [ ]:
a = {"nama": "Budi", "kota": "Jakarta", "tier": "gold"}
b = {"tier": "gold", "nama": "Budi", "kota": "Jakarta"}  # isi sama, urutan beda

print("tanpa sort_keys sama? ", json.dumps(a) == json.dumps(b))
print("dengan sort_keys sama?", json.dumps(a, sort_keys=True) == json.dumps(b, sort_keys=True))

### Cheat code 1: kirim lebih sedikit

**Output tool**: jangan kirim 5.000 baris log, cukup error + ekor.

In [ ]:
log_test = "\n".join(f"PASS test_case_{i}" for i in range(5000))
log_test += "\nFAIL test_login_uppercase_email: expected True, got False"


def ringkas_log(text, tail=50):
    lines = text.splitlines()
    errors = [line for line in lines if "FAIL" in line or "Error" in line]
    return "\n".join(errors + ["..."] + lines[-tail:])


print("log penuh  :", f"{len(enc.encode(log_test)):,}", "token")
print("diringkas  :", f"{len(enc.encode(ringkas_log(log_test))):,}", "token")

**Baca file**: ambil baris yang perlu saja, seperti `read_file(path, offset, limit)` di harness.

In [ ]:
from pathlib import Path

Path("contoh_besar.py").write_text("\n".join(f"# baris {i}" for i in range(1, 2001)))


def read_file(path, offset=0, limit=None):
    lines = Path(path).read_text().splitlines()
    end = None if limit is None else offset + limit
    return "\n".join(lines[offset:end])


print("seluruh file:", f"{len(enc.encode(read_file('contoh_besar.py'))):,}", "token")
print("baris 100-140:", f"{len(enc.encode(read_file('contoh_besar.py', 100, 40))):,}", "token")

**Gambar**: token gambar ikut resolusi. Perkecil sebelum dikirim.

In [ ]:
import base64

from PIL import Image

img = Image.new("RGB", (4000, 3000), "white")
img.save("besar.jpg", quality=90)

kecil = img.copy()
kecil.thumbnail((1024, 1024))
kecil.save("kecil.jpg", quality=85)

for f in ["besar.jpg", "kecil.jpg"]:
    w, h = Image.open(f).size
    b64 = base64.b64encode(Path(f).read_bytes())
    print(f"{f:<10} {w}×{h:<5} base64 {len(b64):>9,} karakter")

## 6. Agent loop: input menumpuk, output cuma tool call

Agent mini yang membenerin bug login. Tools-nya dikunci ke folder `demo_app/` saja.
Perhatikan tabel di akhir: **input membesar tiap giliran**, output kecil, dan sebagian besar input kena cache.

In [ ]:
import subprocess
import sys

APP = Path("demo_app").resolve()
APP.mkdir(exist_ok=True)
(APP / "auth.py").write_text('''USERS = {"eric@devfest.id": "rahasia123"}


def normalize_email(email: str) -> str:
    return email.strip()


def login(email: str, password: str) -> bool:
    stored = USERS.get(normalize_email(email))
    return stored is not None and stored == password
''')
(APP / "test_auth.py").write_text('''import unittest

from auth import login


class TestLogin(unittest.TestCase):
    def test_login_ok(self):
        self.assertTrue(login("eric@devfest.id", "rahasia123"))

    def test_login_uppercase_email(self):
        self.assertTrue(login("ERIC@DevFest.id", "rahasia123"))

    def test_wrong_password(self):
        self.assertFalse(login("eric@devfest.id", "salah"))


if __name__ == "__main__":
    unittest.main()
''')
print(sorted(p.name for p in APP.iterdir()))

In [ ]:
def _safe(path):
    p = (APP / path).resolve()
    if APP not in p.parents and p != APP:
        raise ValueError("path di luar demo_app")
    return p


def tool_list_directory(path="."):
    return "\n".join(sorted(x.name for x in _safe(path).iterdir()))


def tool_read_file(path, offset=0, limit=200):
    lines = _safe(path).read_text().splitlines()
    return "\n".join(lines[offset : offset + limit])


def tool_grep(pattern, path="."):
    hits = []
    for f in sorted(_safe(path).glob("*.py")):
        for n, line in enumerate(f.read_text().splitlines(), 1):
            if pattern.lower() in line.lower():
                hits.append(f"{f.name}:{n}: {line}")
    return "\n".join(hits) or "(tidak ada)"


def tool_edit_file(path, old_text, new_text):
    f = _safe(path)
    text = f.read_text()
    if old_text not in text:
        return "old_text tidak ditemukan"
    f.write_text(text.replace(old_text, new_text, 1))
    return "ok"


def tool_run_tests():
    r = subprocess.run(
        [sys.executable, "-m", "unittest", "-q"],
        cwd=APP, capture_output=True, text=True, timeout=60,
    )
    return (r.stdout + r.stderr)[-2000:]


TOOL_FUNCS = {
    "list_directory": tool_list_directory,
    "read_file": tool_read_file,
    "grep": tool_grep,
    "edit_file": tool_edit_file,
    "run_tests": tool_run_tests,
}


def fn(name, desc, props=None, required=()):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": desc,
            "parameters": {"type": "object", "properties": props or {}, "required": list(required)},
        },
    }


TOOLS = [  # urutan tetap = prefix stabil
    fn("list_directory", "Daftar file di folder.", {"path": {"type": "string"}}),
    fn("read_file", "Baca file, bisa sebagian.",
       {"path": {"type": "string"}, "offset": {"type": "integer"}, "limit": {"type": "integer"}}, ["path"]),
    fn("grep", "Cari teks di file .py.", {"pattern": {"type": "string"}, "path": {"type": "string"}}, ["pattern"]),
    fn("edit_file", "Ganti old_text dengan new_text di file.",
       {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}},
       ["path", "old_text", "new_text"]),
    fn("run_tests", "Jalankan unit test proyek."),
]

In [ ]:
messages = [
    {"role": "system", "content": "Kamu coding agent. Pakai tools untuk membaca, mengedit, dan menjalankan test. Kerjakan sampai test lulus, lalu jelaskan singkat."},
    {"role": "user", "content": "Tolong benerin bug login, testnya gagal."},
]
riwayat = []

for giliran in range(1, 13):
    resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
    msg = resp.choices[0].message
    p, c, o = show_usage(resp, f"giliran {giliran}")
    riwayat.append((giliran, p, c, o))

    if not msg.tool_calls:
        print("\n🤖", msg.content)
        break

    messages.append({
        "role": "assistant",
        "content": msg.content,
        "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
    })
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments or "{}")
        print(f"   ↳ {tc.function.name}({', '.join(f'{k}={v!r}' for k, v in args.items())})"[:110])
        try:
            hasil = TOOL_FUNCS[tc.function.name](**args)
        except Exception as e:
            hasil = f"error: {e}"
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(hasil)})

In [ ]:
import matplotlib.pyplot as plt

g = [r[0] for r in riwayat]
cache = [r[2] for r in riwayat]
baru = [r[1] - r[2] for r in riwayat]
keluar = [r[3] for r in riwayat]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(g, cache, color="#d2e3fc", label="input dari cache")
ax.bar(g, baru, bottom=cache, color="#34A853", label="input baru")
ax.bar([x + 0.35 for x in g], keluar, width=0.3, color="#a142f4", label="output")
ax.set_xlabel("giliran")
ax.set_ylabel("token")
ax.set_title("Input menumpuk, output cuma tool call")
ax.legend()
plt.show()

total_in = sum(r[1] for r in riwayat)
total_cache = sum(r[2] for r in riwayat)
total_out = sum(keluar)
print(f"total input {total_in:,} (cache {total_cache / total_in:.0%}), total output {total_out:,}")

## 7. Kuota sadar cache

Slide **Kuota Tinggal 8%, Masih Bisa?**: kuota 10 jt, sudah terpakai 92%, request berikutnya konteks 1 jt token dengan 92% dari cache. Token cache dihitung 10%.

In [ ]:
KUOTA = 10_000_000
terpakai = 9_200_000
sisa = KUOTA - terpakai

konteks = 1_000_000
rasio_cache = 0.92
bobot_cache = 0.10

naif = konteks
sadar_cache = konteks * (1 - rasio_cache) + konteks * rasio_cache * bobot_cache

print(f"sisa kuota      : {sisa:>9,.0f}")
print(f"estimasi naif   : {naif:>9,.0f}  → {'❌ ditolak' if naif > sisa else '✅ lolos'}")
print(f"sadar cache     : {sadar_cache:>9,.0f}  → {'❌ ditolak' if sadar_cache > sisa else '✅ lolos'}")
print(f"masih muat ±{sisa // sadar_cache:.0f} request lagi")

## Ringkasan

1. Token itu satuan biaya; **input** yang paling banyak, terutama di agent.
2. Tiga cheat code: **kirim lebih sedikit**, **jangan bayar dua kali** (cache), **bayar lebih murah per token** (pilih model).
3. Cache cocok **dari depan** dan punya **umur**. Yang stabil di depan, yang berubah di belakang.
4. Ukur `cached_tokens` di setiap request.

Klaim kuota: **platform.orvix.id/e/ROAD-TO-DEVFEST-JKT-2026**